# Data Preparation: Rogers Pass Snow Profiles

This notebook is the **source-of-truth** record of the non-destructive preparation
of the 2025 Rogers Pass field campaign for FRDR deposit. Edit cells in place via
NotebookEdit; there is no driver script.

**Inputs**
- `datasets/rogers_pass_snow_profiles/raw_data/Rogers Pass March 2024-2025/`

**Outputs**
- `datasets/rogers_pass_snow_profiles/frdr_data/` — the prepared deposit package.
- Markdown tables (transformation log + per-category summary) printed by this
  notebook, ready to paste into `artifacts/data_preparation_report.md`.

**Naming conventions**
- Standardized folder names use `<datatype>/<YYYYMMDD>_<site>` or
  `<datatype>/<YYYYMMDD>_<site>_<content>`.
- Standardized top-level file names use `<YYYYMMDD>_<site>_<content>.<ext>`.
- Files inside bulk folders keep their instrument-native names (lowercased).

**Non-destructive guarantee.** Scientific file contents are never edited. Only
package structure, file names, and DOCX → TXT conversion (for ancillary field
notes) are performed. The Day 5 raw `radar_ka` folder is reorganized under the
generic `radar/` deposit folder; the band terminology question is documented in
the QC report rather than altered in the data.

Reusable helpers live in `utils/preparation.py`.

## Setup

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path so `utils.preparation` is importable
# regardless of where the kernel was started.
notebook_dir = Path.cwd()
for candidate in [notebook_dir, *notebook_dir.parents]:
    if (candidate / "utils" / "preparation.py").exists():
        project_root = candidate
        break
else:
    raise RuntimeError("Could not locate project root containing utils/preparation.py")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.preparation import (
    TransformLog,
    copy_and_rename,
    copy_directory_files,
    docx_to_txt,
)

DATASET_ROOT = project_root / "datasets" / "rogers_pass_snow_profiles"
RAW_ROOT = DATASET_ROOT / "raw_data" / "Rogers Pass March 2024-2025"
FRDR_ROOT = DATASET_ROOT / "frdr_data"

log = TransformLog()
print("project_root:", project_root)
print("RAW_ROOT exists:", RAW_ROOT.exists())
print("FRDR_ROOT:", FRDR_ROOT)

project_root: c:\Users\beav3503\dev\grimp_frdr_helper
RAW_ROOT exists: True
FRDR_ROOT: c:\Users\beav3503\dev\grimp_frdr_helper\datasets\rogers_pass_snow_profiles\frdr_data


## Reset target tree

Clear `frdr_data/` so the notebook is idempotent. Guarded with `RESET = True`.
Set to `False` to skip the reset (useful when partially rerunning cells without
rebuilding the whole package).

In [2]:
import shutil

RESET = True
if RESET and FRDR_ROOT.exists():
    shutil.rmtree(FRDR_ROOT)
FRDR_ROOT.mkdir(parents=True, exist_ok=True)
print("frdr_data/ ready:", FRDR_ROOT.exists(), "— contents:", sorted(p.name for p in FRDR_ROOT.iterdir()))

frdr_data/ ready: True — contents: []


## Single-file rename and reorganize

Twenty-three files are copied with explicit `<source, target>` pairs into the
prepared folder structure. Stratigraphy workbooks, IRIS daily exports, spatial
linkage workbooks, GPS CSV exports, and shapefile ZIPs.

In [3]:
single_file_mappings = [
    # Snow stratigraphy workbooks (one per site-day)
    (RAW_ROOT / "Jour 1 - Fidelity" / "Strati_20250301_fidelity.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250301_fidelity_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "20250302_JimBay_StratiTemplate.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250302_jim_bay_corner_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 3 - Hermit" / "20250303_Hermit.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250303_hermit_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 4 - Fidelity" / "20250304_StratiTemplate.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250304_fidelity_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "20250305_Strati.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250305_round_hill_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "CRidge_Strati_20250306.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250306_christiana_ridge_stratigraphy.xlsx"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "Fidelity_Strati_20250306.xlsx",
     FRDR_ROOT / "snow_stratigraphy" / "20250306_fidelity_revisit_stratigraphy.xlsx"),
    # IRIS daily raw text exports
    (RAW_ROOT / "Jour 1 - Fidelity" / "IRIS data" / "IRIS_20250301.TXT",
     FRDR_ROOT / "iris" / "20250301_fidelity_iris_raw.txt"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "IRIS" / "20250302.TXT",
     FRDR_ROOT / "iris" / "20250302_jim_bay_corner_iris_raw.txt"),
    (RAW_ROOT / "Jour 3 - Hermit" / "IRIS" / "20250303.TXT",
     FRDR_ROOT / "iris" / "20250303_hermit_iris_raw.txt"),
    (RAW_ROOT / "Jour 4 - Fidelity" / "IRIS" / "20250304.TXT",
     FRDR_ROOT / "iris" / "20250304_fidelity_iris_raw.txt"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "IRIS" / "20250305.TXT",
     FRDR_ROOT / "iris" / "20250305_round_hill_iris_raw.txt"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "IRIS" / "20250306.TXT",
     FRDR_ROOT / "iris" / "20250306_christiana_ridge_iris_raw.txt"),
    # Spatial linkage workbooks
    (RAW_ROOT / "Jour 1 - Fidelity" / "Fidelity Radars and SMP" / "Radar Fidelity point information.xlsx",
     FRDR_ROOT / "spatial_reference" / "20250301_fidelity_spatial_linkage.xlsx"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "Spatial Survey - SMP and Radar K" / "20250302_JimBayCornerSpatial.xlsx",
     FRDR_ROOT / "spatial_reference" / "20250302_jim_bay_corner_spatial_linkage.xlsx"),
    (RAW_ROOT / "Jour 3 - Hermit" / "Spatial_Hermitt" / "20250303_HermitWX_RadarK.xlsx",
     FRDR_ROOT / "spatial_reference" / "20250303_hermit_spatial_linkage.xlsx"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "Spatial Survey notes.xlsx",
     FRDR_ROOT / "spatial_reference" / "20250305_round_hill_spatial_linkage.xlsx"),
    # GPS CSV exports
    (RAW_ROOT / "Jour 1 - Fidelity" / "Fidelity Radars and SMP" / "Radar Fidelity.csv",
     FRDR_ROOT / "spatial_reference" / "20250301_fidelity_gps_points.csv"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "Spatial Survey - SMP and Radar K" / "jimbaycorner_01032025.csv",
     FRDR_ROOT / "spatial_reference" / "20250302_jim_bay_corner_gps_points.csv"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "GPS information" / "ROUND HILL.csv",
     FRDR_ROOT / "spatial_reference" / "20250305_round_hill_gps_points.csv"),
    # Shapefile ZIPs
    (RAW_ROOT / "Jour 1 - Fidelity" / "Fidelity Radars and SMP" / "Radar Fidelity.shp.zip",
     FRDR_ROOT / "spatial_reference" / "20250301_fidelity_gps_points_shapefile.zip"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "Spatial Survey - SMP and Radar K" / "jimbaycorner_01032025.shp.zip",
     FRDR_ROOT / "spatial_reference" / "20250302_jim_bay_corner_gps_points_shapefile.zip"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "GPS information" / "ROUND HILL.shp.zip",
     FRDR_ROOT / "spatial_reference" / "20250305_round_hill_gps_points_shapefile.zip"),
]

for src, dest in single_file_mappings:
    category = dest.relative_to(FRDR_ROOT).parts[0]
    log.add(copy_and_rename(src, dest, category=category, relative_to=DATASET_ROOT))

print(f"copy_and_rename ops: {len(single_file_mappings)}")

copy_and_rename ops: 23


## Bulk folder copies (snowscope, SMP, radar)

Each (source folder, destination folder, glob) tuple copies every matching file
into the standardized deposit folder. File names are lowercased by the helper.
The Day 6 SnowScope raw folder name typo (`SS4_christridge_20240306`) is
corrected here in the deposit folder name only — file contents are unchanged.

In [4]:
directory_mappings = [
    # SnowScope (CSV)
    (RAW_ROOT / "Jour 1 - Fidelity" / "Spatial_Jimbaycorner" / "SS_SMP_20250301",
     FRDR_ROOT / "snowscope" / "20250301_jim_bay_corner_snowscope_smp", "*.csv"),
    (RAW_ROOT / "Jour 1 - Fidelity" / "Spatial_Jimbaycorner" / "SS_SS1_20250301",
     FRDR_ROOT / "snowscope" / "20250301_jim_bay_corner_snowscope", "*.csv"),
    (RAW_ROOT / "Jour 3 - Hermit" / "Spatial_Hermitt" / "SS_SS2_hermitt_20250303",
     FRDR_ROOT / "snowscope" / "20250303_hermit_snowscope", "*.csv"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "SS_SN322_20250306",
     FRDR_ROOT / "snowscope" / "20250305_round_hill_snowscope", "*.csv"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "Spatial Survey Christiana Ridge" / "SS4_christridge_20240306",
     FRDR_ROOT / "snowscope" / "20250306_christiana_ridge_snowscope", "*.csv"),
    # Radar (TXT) — Phase 1 keeps the folder name `radar/`
    (RAW_ROOT / "Jour 1 - Fidelity" / "Spatial_Jimbaycorner" / "radar_k",
     FRDR_ROOT / "radar" / "20250301_jim_bay_corner", "*.txt"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "Spatial Survey - SMP and Radar K" / "radar_k",
     FRDR_ROOT / "radar" / "20250302_jim_bay_corner", "*.txt"),
    (RAW_ROOT / "Jour 3 - Hermit" / "Spatial_Hermitt" / "radar_k",
     FRDR_ROOT / "radar" / "20250303_hermit", "*.txt"),
    (RAW_ROOT / "Jour 4 - Fidelity" / "Radar K",
     FRDR_ROOT / "radar" / "20250304_fidelity", "*.txt"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "radar_ka",
     FRDR_ROOT / "radar" / "20250305_round_hill", "*.txt"),
    # SnowMicroPenetrometer (.pnt)
    (RAW_ROOT / "Jour 1 - Fidelity" / "Fidelity Radars and SMP",
     FRDR_ROOT / "snowmicropenetrometer" / "20250301_fidelity", "*.pnt"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "Spatial Survey - SMP and Radar K" / "SMP",
     FRDR_ROOT / "snowmicropenetrometer" / "20250302_jim_bay_corner", "*.pnt"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "Spatial Survey" / "SMP",
     FRDR_ROOT / "snowmicropenetrometer" / "20250305_round_hill", "*.pnt"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "SMP",
     FRDR_ROOT / "snowmicropenetrometer" / "20250306_christiana_ridge", "*.pnt"),
]

bulk_total = 0
for src_dir, dest_dir, pattern in directory_mappings:
    category = dest_dir.relative_to(FRDR_ROOT).parts[0]
    records = copy_directory_files(
        src_dir, dest_dir, category=category, relative_to=DATASET_ROOT, pattern=pattern
    )
    log.extend(records)
    bulk_total += len(records)
    print(f"  {src_dir.name} -> {dest_dir.relative_to(FRDR_ROOT)}  ({len(records)} files)")

print(f"copy_and_restructure ops: {bulk_total}")

  SS_SMP_20250301 -> snowscope\20250301_jim_bay_corner_snowscope_smp  (8 files)


  SS_SS1_20250301 -> snowscope\20250301_jim_bay_corner_snowscope  (43 files)


  SS_SS2_hermitt_20250303 -> snowscope\20250303_hermit_snowscope  (59 files)


  SS_SN322_20250306 -> snowscope\20250305_round_hill_snowscope  (98 files)


  SS4_christridge_20240306 -> snowscope\20250306_christiana_ridge_snowscope  (111 files)


  radar_k -> radar\20250301_jim_bay_corner  (50 files)


  radar_k -> radar\20250302_jim_bay_corner  (42 files)
  radar_k -> radar\20250303_hermit  (28 files)
  Radar K -> radar\20250304_fidelity  (2 files)


  radar_ka -> radar\20250305_round_hill  (44 files)
  Fidelity Radars and SMP -> snowmicropenetrometer\20250301_fidelity  (4 files)
  SMP -> snowmicropenetrometer\20250302_jim_bay_corner  (18 files)


  SMP -> snowmicropenetrometer\20250305_round_hill  (46 files)
  SMP -> snowmicropenetrometer\20250306_christiana_ridge  (11 files)
copy_and_restructure ops: 564


## DOCX → TXT conversion (field and instrument notes)

Eight DOCX field/instrument notes are converted to UTF-8 plain text using
pandoc. Original French content is preserved; only formatting is dropped.
These are tagged role=`support` to distinguish them from scientific files.

In [5]:
docx_mappings = [
    (RAW_ROOT / "Jour 1 - Fidelity" / "20250301_FidelityWx-JimBayCorner_ReadMe.docx",
     FRDR_ROOT / "documentation" / "20250301_fidelity_jim_bay_corner_field_notes_fr.txt"),
    (RAW_ROOT / "Jour 1 - Fidelity" / "Fidelity Radars and SMP" / "Read_me_manipSMP_Fidelity.docx",
     FRDR_ROOT / "documentation" / "20250301_fidelity_smp_setup_notes_fr.txt"),
    (RAW_ROOT / "Jour 2 - Jim Bay" / "20250302_JimBayCorner_ReadMe.docx",
     FRDR_ROOT / "documentation" / "20250302_jim_bay_corner_field_notes_fr.txt"),
    (RAW_ROOT / "Jour 3 - Hermit" / "20250303_HermitWx_ReadMe.docx",
     FRDR_ROOT / "documentation" / "20250303_hermit_field_notes_fr.txt"),
    (RAW_ROOT / "Jour 4 - Fidelity" / "20250304_Fidelity.docx",
     FRDR_ROOT / "documentation" / "20250304_fidelity_field_notes_fr.txt"),
    (RAW_ROOT / "Jour 4 - Fidelity" / "Radar K" / "20250304_radar_readme.docx",
     FRDR_ROOT / "documentation" / "20250304_fidelity_radar_notes_fr.txt"),
    (RAW_ROOT / "Jour 5 - Round Hill" / "20250305_RoundHill_ReadMe.docx",
     FRDR_ROOT / "documentation" / "20250305_round_hill_field_notes_fr.txt"),
    (RAW_ROOT / "Jour 6 - RoundHill and Christiana Ridge" / "20250306_RoundHill et Christiana_.docx",
     FRDR_ROOT / "documentation" / "20250306_round_hill_christiana_ridge_field_notes_fr.txt"),
]

for src, dest in docx_mappings:
    log.add(docx_to_txt(
        src, dest,
        category="documentation",
        relative_to=DATASET_ROOT,
        cwd=project_root,
    ))

print(f"docx_to_txt ops: {len(docx_mappings)}")

docx_to_txt ops: 8


## Excluded files (with reasons)

The campaign raw folder includes administrative material, blank templates,
DOCX duplicates, hazard PDFs, and HEIC field-notebook photos that are *not*
deposited. Exclusions are recorded explicitly here so the deposit boundary is
reviewable.

In [6]:
included_sources = {r.source for r in log.records}
raw_files = sorted(p for p in RAW_ROOT.rglob("*") if p.is_file())

def classify_excluded(filepath):
    name = filepath.name
    parts = filepath.parts
    if filepath.suffix.lower() == ".heic":
        return "Excluded field notebook photo/image pending researcher decision"
    if "Bouffe et infos" in parts:
        return "Excluded logistics spreadsheet"
    if name == "StratiTemplate.xlsx":
        return "Excluded blank template workbook"
    if "Hazard" in name:
        return "Excluded hazard assessment PDF pending researcher decision"
    if name == "YUL parking reservation.pdf":
        return "Excluded travel logistics PDF"
    if name.startswith("www.expedia"):
        return "Excluded travel booking PDF"
    if name == "Campagne Roger_s Pass 2025 planification.docx":
        return "Excluded planning document"
    if name == "Table of content Field Books.docx":
        return "Excluded internal index document"
    if name == "IRIS_20250301.TXT.docx":
        return "Excluded companion DOCX duplicate of IRIS raw text"
    return "Excluded administrative or unresolved ancillary material"

excluded = []
for fp in raw_files:
    rel = fp.relative_to(DATASET_ROOT).as_posix()
    if rel in included_sources:
        continue
    excluded.append({"source": rel, "reason": classify_excluded(fp)})

from collections import Counter
reason_counts = Counter(e["reason"] for e in excluded)
print(f"raw_total_files:      {len(raw_files)}")
print(f"prepared_total_files: {len(log)}")
print(f"excluded_total_files: {len(excluded)}")
print()
print("Excluded reason counts:")
for reason, count in reason_counts.most_common():
    print(f"  {count:3d}  {reason}")

raw_total_files:      633
prepared_total_files: 595
excluded_total_files: 38

Excluded reason counts:
   25  Excluded field notebook photo/image pending researcher decision
    6  Excluded hazard assessment PDF pending researcher decision
    1  Excluded logistics spreadsheet
    1  Excluded planning document
    1  Excluded companion DOCX duplicate of IRIS raw text
    1  Excluded travel booking PDF
    1  Excluded blank template workbook
    1  Excluded internal index document
    1  Excluded travel logistics PDF


## Structural statistics on prepared files

Per-category structural stats (file counts, ranges) computed inline. These are
used in the README and the data preparation report. No scientific values are
modified; this cell only reads the prepared files.

In [7]:
from statistics import mean
import openpyxl
import pandas as pd
import snowmicropyn

stats = {}

# snow_stratigraphy
strati_files = sorted((FRDR_ROOT / "snow_stratigraphy").glob("*.xlsx"))
strati_elevations, strati_temperatures = [], []
for fp in strati_files:
    wb = openpyxl.load_workbook(fp, data_only=True, read_only=True)
    ws = wb["AVY profile"]
    for row in ws.iter_rows(values_only=True):
        for v in row:
            if isinstance(v, (int, float)):
                if 1500 <= float(v) <= 2500:
                    strati_elevations.append(float(v))
                if -40 <= float(v) <= 2:
                    strati_temperatures.append(float(v))
    wb.close()
stats["snow_stratigraphy"] = {
    "file_count": len(strati_files),
    "sheet_count_per_file": 4,
    "elevation_range_m": [min(strati_elevations), max(strati_elevations)],
    "temperature_range_c": [min(strati_temperatures), max(strati_temperatures)],
}

# iris
iris_files = sorted((FRDR_ROOT / "iris").glob("*.txt"))
iris_row_counts, iris_values = [], []
for fp in iris_files:
    df = pd.read_csv(fp, header=None, names=["time", "value"])
    iris_row_counts.append(len(df))
    iris_values.extend(df["value"].dropna().astype(float).tolist())
stats["iris"] = {
    "file_count": len(iris_files),
    "row_range": [min(iris_row_counts), max(iris_row_counts)],
    "value_range": [min(iris_values), max(iris_values)],
}

# snowmicropenetrometer
smp_files = sorted((FRDR_ROOT / "snowmicropenetrometer").rglob("*.pnt"))
smp_samples, smp_depths, smp_forces, smp_serials = [], [], [], set()
for fp in smp_files:
    profile = snowmicropyn.Profile.load(str(fp))
    samples = profile.samples
    smp_samples.append(len(samples))
    smp_depths.append(float(samples["distance"].max()))
    smp_forces.append(float(samples["force"].max()))
    smp_serials.add(profile.smp_serial)
stats["snowmicropenetrometer"] = {
    "file_count": len(smp_files),
    "serials": sorted(smp_serials),
    "sample_range": [min(smp_samples), max(smp_samples)],
    "depth_range_mm": [min(smp_depths), max(smp_depths)],
    "force_range_n": [round(min(smp_forces), 2), round(max(smp_forces), 2)],
}

# snowscope
def parse_snowscope_csv(filepath):
    text = filepath.read_text(encoding="utf-8")
    lines = text.strip().splitlines()
    meta, data_start = {}, None
    for i, line in enumerate(lines):
        if line.startswith("depth (mm)"):
            data_start = i
            break
        if "," in line:
            k, v = line.split(",", 1)
            if k.strip():
                meta[k.strip()] = v.strip()
    if data_start is None:
        return meta, pd.DataFrame()
    header = [c.strip() for c in lines[data_start].split(",")]
    rows = [line.split(",") for line in lines[data_start + 1:] if line.strip()]
    df = pd.DataFrame(rows, columns=header)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return meta, df

snowscope_files = sorted((FRDR_ROOT / "snowscope").rglob("*.csv"))
ss_depths, ss_rows, ss_serials, ss_fw, ss_pcb, ss_locs = [], [], set(), set(), set(), []
with_optical = without_optical = 0
for fp in snowscope_files:
    meta, df = parse_snowscope_csv(fp)
    if "profileDepth (mm)" in meta:
        try:
            ss_depths.append(float(meta["profileDepth (mm)"]))
        except ValueError:
            pass
    if "serialNum" in meta:
        ss_serials.add(meta["serialNum"])
    if "FW_version" in meta:
        ss_fw.add(meta["FW_version"])
    if "PCB_version" in meta:
        ss_pcb.add(meta["PCB_version"])
    if meta.get("Location", "null") != "null":
        try:
            lat, lon = [float(v) for v in meta["Location"].split(",")]
            ss_locs.append((lat, lon))
        except ValueError:
            pass
    ss_rows.append(len(df))
    if "optical Reflectance Avg" in df.columns:
        with_optical += 1
    else:
        without_optical += 1
stats["snowscope"] = {
    "file_count": len(snowscope_files),
    "serials": sorted(ss_serials),
    "firmware_versions": sorted(ss_fw),
    "pcb_versions": sorted(ss_pcb),
    "row_range": [min(ss_rows), max(ss_rows)],
    "profile_depth_range_mm": [min(ss_depths), max(ss_depths)],
    "with_optical": with_optical,
    "without_optical": without_optical,
    "lat_range": [min(lat for lat, _ in ss_locs), max(lat for lat, _ in ss_locs)],
    "lon_range": [min(lon for _, lon in ss_locs), max(lon for _, lon in ss_locs)],
}

# radar
def parse_radar_txt(filepath):
    text = filepath.read_text(encoding="utf-8")
    lines = text.strip().splitlines()
    meta, data_start = {}, None
    for i, line in enumerate(lines):
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if s.startswith("X (m)"):
            data_start = i
            break
        if ":" in s:
            k, v = s.split(":", 1)
            meta[k.strip()] = v.strip()
    if data_start is None:
        return meta, pd.DataFrame()
    header = [c.strip() for c in lines[data_start].split(",")]
    rows = []
    for line in lines[data_start + 1:]:
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        parts = [p.strip() for p in s.split(",")]
        if len(parts) != len(header):
            continue
        try:
            rows.append([float(p) for p in parts])
        except ValueError:
            continue
    return meta, pd.DataFrame(rows, columns=header)

radar_files = sorted((FRDR_ROOT / "radar").rglob("*.txt"))
radar_rows, radar_numbers, radar_freqs = [], set(), set()
for fp in radar_files:
    meta, df = parse_radar_txt(fp)
    radar_rows.append(len(df))
    if "Radar No." in meta:
        radar_numbers.add(meta["Radar No."])
    if "Start-Frequency [MHz]" in meta and "Stop-Frequency [MHz]" in meta:
        radar_freqs.add((meta["Start-Frequency [MHz]"], meta["Stop-Frequency [MHz]"]))
stats["radar"] = {
    "file_count": len(radar_files),
    "row_range": [min(radar_rows), max(radar_rows)],
    "radar_numbers": sorted(radar_numbers),
    "frequency_ranges_mhz": sorted(radar_freqs),
}

# spatial_reference
gps_files = sorted((FRDR_ROOT / "spatial_reference").glob("*_gps_points.csv"))
gps_lats, gps_lons, gps_elevs = [], [], []
for fp in gps_files:
    df = pd.read_csv(fp)
    lat_col = "Latitude" if "Latitude" in df.columns else None
    lon_col = "Longitude" if "Longitude" in df.columns else None
    elev_col = "Elevation" if "Elevation" in df.columns else None
    if elev_col and int(df[elev_col].notna().sum()) == 0 and "Ellipsoidal height" in df.columns:
        elev_col = "Ellipsoidal height"
    if lat_col:
        gps_lats.extend(df[lat_col].dropna().astype(float).tolist())
    if lon_col:
        gps_lons.extend(df[lon_col].dropna().astype(float).tolist())
    if elev_col:
        gps_elevs.extend(df[elev_col].dropna().astype(float).tolist())
stats["spatial_reference"] = {
    "gps_csv_count": len(gps_files),
    "shapefile_zip_count": len(list((FRDR_ROOT / "spatial_reference").glob("*_shapefile.zip"))),
    "linkage_workbook_count": len(list((FRDR_ROOT / "spatial_reference").glob("*_spatial_linkage.xlsx"))),
    "lat_range": [min(gps_lats), max(gps_lats)],
    "lon_range": [min(gps_lons), max(gps_lons)],
    "elevation_range_m": [min(gps_elevs), max(gps_elevs)],
}

# documentation
doc_files = sorted((FRDR_ROOT / "documentation").glob("*.txt"))
stats["documentation"] = {
    "file_count": len(doc_files),
    "language": "French originals converted from DOCX to UTF-8 plain text",
}

# dataset bbox combines GPS + SnowScope locations
lat_values = gps_lats + [lat for lat, _ in ss_locs]
lon_values = gps_lons + [lon for _, lon in ss_locs]
all_elevs = gps_elevs + strati_elevations
stats["dataset_bbox"] = {
    "north": max(lat_values),
    "south": min(lat_values),
    "east": max(lon_values),
    "west": min(lon_values),
    "mean_latitude": mean([max(lat_values), min(lat_values)]),
    "mean_longitude": mean([max(lon_values), min(lon_values)]),
    "elevation_min_m": min(all_elevs),
    "elevation_max_m": max(all_elevs),
}

stats

Latitude value -99999.0 invalid, replacing by None (file S35M0129)


Longitude value None invalid, replacing by None (file S35M0129)


Latitude value -99999.0 invalid, replacing by None (file S35M0131)


Longitude value None invalid, replacing by None (file S35M0131)


Latitude value -99999.0 invalid, replacing by None (file S35M0133)


...

Latitude value -99999.0 invalid, replacing by None (file S35M0196)


Longitude value None invalid, replacing by None (file S35M0196)


Latitude value -99999.0 invalid, replacing by None (file S35M0205)


Longitude value None invalid, replacing by None (file S35M0205)


{'snow_stratigraphy': {'file_count': 7,
  'sheet_count_per_file': 4,
  'elevation_range_m': [1875.0, 2088.0],
  'temperature_range_c': [-8.5, 2.0]},
 'iris': {'file_count': 6,
  'row_range': [52, 92],
  'value_range': [0.006, 2.002]},
 'snowmicropenetrometer': {'file_count': 79,
  'serials': ['35'],
  'sample_range': [3, 411400],
  'depth_range_mm': [0.00826446246355772, 1699.9957965225913],
  'force_range_n': [0.03, 41.92]},
 'snowscope': {'file_count': 319,
  'serials': ['00304', '00322', '00328'],
  'firmware_versions': ['2.4.1'],
  'pcb_versions': ['v2.7'],
  'row_range': [81, 2266],
  'profile_depth_range_mm': [81.0, 2266.0],
  'with_optical': 181,
  'without_optical': 138,
  'lat_range': [51.2341198, 51.3236271],
  'lon_range': [-117.713247, -117.5314499]},
 'radar': {'file_count': 166,
  'row_range': [2565, 2565],
  'radar_numbers': ['2010000058'],
  'frequency_ranges_mhz': [('23500', '26000')]},
 'spatial_reference': {'gps_csv_count': 3,
  'shapefile_zip_count': 3,
  'linkage_w

## Render report tables

Markdown blocks below are the inventory tables. Copy them into
`artifacts/data_preparation_report.md` (or render them in place under
explicit markers).

In [8]:
print("### Per-category summary\n")
print(log.category_summary_markdown())

### Per-category summary

| Category | Files | Size (MB) | Roles |
|---|---:|---:|---|
| documentation | 8 | 0.01 | support: 8 |
| iris | 6 | 0.01 | scientific: 6 |
| radar | 166 | 17.82 | scientific: 166 |
| snow_stratigraphy | 7 | 1.51 | scientific: 7 |
| snowmicropenetrometer | 79 | 62.35 | scientific: 79 |
| snowscope | 319 | 10.01 | scientific: 319 |
| spatial_reference | 10 | 0.12 | scientific: 10 |


In [9]:
print("### Transformation log (first 30 rows shown — full table is", len(log), "rows)\n")
full = log.to_markdown_table().splitlines()
# Header (2 lines) + first 30 data rows
print("\n".join(full[: 2 + 30]))

### Transformation log (first 30 rows shown — full table is 595 rows)

| Category | Transform | Source | Target |
|---|---|---|---|
| snow_stratigraphy | copy_and_rename | `raw_data/Rogers Pass March 2024-2025/Jour 1 - Fidelity/Strati_20250301_fidelity.xlsx` | `frdr_data/snow_stratigraphy/20250301_fidelity_stratigraphy.xlsx` |
| snow_stratigraphy | copy_and_rename | `raw_data/Rogers Pass March 2024-2025/Jour 2 - Jim Bay/20250302_JimBay_StratiTemplate.xlsx` | `frdr_data/snow_stratigraphy/20250302_jim_bay_corner_stratigraphy.xlsx` |
| snow_stratigraphy | copy_and_rename | `raw_data/Rogers Pass March 2024-2025/Jour 3 - Hermit/20250303_Hermit.xlsx` | `frdr_data/snow_stratigraphy/20250303_hermit_stratigraphy.xlsx` |
| snow_stratigraphy | copy_and_rename | `raw_data/Rogers Pass March 2024-2025/Jour 4 - Fidelity/20250304_StratiTemplate.xlsx` | `frdr_data/snow_stratigraphy/20250304_fidelity_stratigraphy.xlsx` |
| snow_stratigraphy | copy_and_rename | `raw_data/Rogers Pass March 2024-2025/Jour 

## Sanity report

In [10]:
summary = log.summary()
bbox = stats["dataset_bbox"]
top_level = sorted(p.name for p in FRDR_ROOT.iterdir() if p.is_dir())

print(f"prepared_total_files: {summary['total_files']}")
print(f"excluded_total_files: {len(excluded)}")
print(f"transform_counts:     {summary['transform_counts']}")
print(f"top-level folders:    {top_level}")
print(f"bbox: N={bbox['north']:.6f}  S={bbox['south']:.6f}  E={bbox['east']:.6f}  W={bbox['west']:.6f}")
print(f"elevation: {bbox['elevation_min_m']:.0f}-{bbox['elevation_max_m']:.0f} m")

expected = {"copy_and_rename": 23, "copy_and_restructure": 564, "docx_to_txt": 8}
assert summary["transform_counts"] == expected, (summary["transform_counts"], expected)
assert summary["total_files"] == 595
assert len(excluded) == 38
print("\nAll structural assertions passed.")

prepared_total_files: 595
excluded_total_files: 38
transform_counts:     {'copy_and_rename': 23, 'copy_and_restructure': 564, 'docx_to_txt': 8}
top-level folders:    ['documentation', 'iris', 'radar', 'snow_stratigraphy', 'snowmicropenetrometer', 'snowscope', 'spatial_reference']
bbox: N=51.323627  S=51.234087  E=-117.531450  W=-117.713247
elevation: 1832-2088 m

All structural assertions passed.
